# Staged Readout Amplitude Optimizer

This notebook runs the dependency-safe submit-only readout optimizer flow.

The optimizer is split into two stages:

1. Submit kernel-trace experiments for every readout amplitude.
2. After kernels finish, collect/analyze them, save amplitude-specific kernel files, submit IQ blobs with the matching kernels, then collect IQ results and save the optimizer output.

Do not submit kernels and IQ blobs together in submit-only mode. IQ blobs depend on the generated kernel files.

## Configuration

Edit this cell before running. For Stage 1, leave `RUN_KEY` empty. For Stage 2 or collection, paste either the run folder name from `metadata.json` or the full run folder path.

In [38]:
from pathlib import Path

import numpy as np

PROFILE_NAME = "main"
PROFILE_BRANCH = None  # None means use PROFILE_NAME
QUBIT_NAMES = ["q1","q3", "q4", "q5", "q6", "q7", "q8","q9", "q10", "q11", "q12", "q13", "q14", "q15", "q16", "q17", "q18", "q19"]
AMPLITUDES = np.linspace(0.005, 0.15, 25)

OUTPUT_ROOT = Path("data") / "readout_optimize"
RUN_KEY = None  # filled automatically by Stage 1; set manually only when resuming an existing run

LOW_PRIORITY_TASKS = True
WAIT_FOR_RESULTS = False
STATES = ["g", "e"]
RESET_NUM = 5


## Imports And Helpers

In [39]:
import json
import sys

WORKBENCH_ROOT = Path.cwd()
if str(WORKBENCH_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKBENCH_ROOT))

from qratena.system.components_params.reset_settings import ResetSettings
from qratena.util.enums import ResetType

from optimize.readout.readout_amplitude_optimizer import (
    ReadoutAmplitudeSweepSettings,
    ReadoutAmplitudeSweepWorkflow,
)
from optimize.readout.readout_workflow import ReadoutFidelityWorkflowSettings
from optimize.readout.utils.readout_scan_types import ReadoutScanMethod
from optimize.readout.utils.readout_sweep_artifacts import load_readout_task_manifest
from resources.load_profile import load_profile, load_task_manager


In [40]:
def resolve_run_dir(run_key: str | Path | None = None, output_root: Path = OUTPUT_ROOT) -> Path:
    if run_key is None:
        run_key = RUN_KEY

    if not run_key:
        raise ValueError("Run Stage 1 first, or set RUN_KEY to an existing run folder name/full path.")

    candidate = Path(run_key).expanduser()
    if candidate.exists():
        return candidate.resolve()

    matches = [
        manifest_path.parent
        for manifest_path in output_root.expanduser().rglob("task_manifest.json")
        if manifest_path.parent.name == run_key
    ]
    if len(matches) == 1:
        return matches[0].resolve()
    if not matches:
        raise FileNotFoundError(f"Could not find run folder {run_key!r} under {output_root}.")

    joined = "\n".join(str(path) for path in matches)
    raise RuntimeError(
        f"Run key {run_key!r} matched multiple folders:\n{joined}\n"
        "Use the full run folder path."
    )


def load_metadata(run_dir: Path) -> dict:
    metadata_path = run_dir / "metadata.json"
    if not metadata_path.exists():
        return {}
    return json.loads(metadata_path.read_text(encoding="utf-8"))


def is_staged_kernel_run(manifest: dict) -> bool:
    tasks = list(manifest.get("tasks", []))
    return bool(tasks) and any(task.get("node") == "kernels" for task in tasks) and not any(
        task.get("node") == "iq_blobs" for task in tasks
    )


def is_staged_iq_run(manifest: dict) -> bool:
    return any(
        task.get("stage") == "iq_blobs" or task.get("depends_on_stage") == "kernels"
        for task in manifest.get("tasks", [])
    )


def print_task_keys(title: str, task_keys: list[str | None]) -> None:
    if not task_keys:
        return
    print(f"{title}:")
    for task_key in task_keys:
        print(f"  - {task_key}")


In [41]:
def make_workflow_settings(
    *,
    run_kernels: bool,
    run_iq_blobs: bool,
    low_priority_tasks: bool = LOW_PRIORITY_TASKS,
) -> ReadoutFidelityWorkflowSettings:
    return ReadoutFidelityWorkflowSettings(
        profile_name=PROFILE_NAME,
        do_emulation=False,
        run_resonator=False,
        run_kernels=run_kernels,
        run_iq_blobs=run_iq_blobs,
        do_plotting=False,
        show_handler_output=False,
        low_priority_tasks=low_priority_tasks,
        states=list(STATES),
        reset=ResetSettings(reset_type = ResetType.ACTIVE, reset_num=RESET_NUM),
    )


def make_optimizer(
    *,
    amplitudes,
    run_kernels: bool,
    run_iq_blobs: bool,
    submit_only: bool = False,
) -> ReadoutAmplitudeSweepWorkflow:
    profile = load_profile(PROFILE_BRANCH or PROFILE_NAME)
    task_manager = load_task_manager()
    workflow_settings = make_workflow_settings(
        run_kernels=run_kernels,
        run_iq_blobs=run_iq_blobs,
    )
    optimizer_settings = ReadoutAmplitudeSweepSettings(
        amplitudes=amplitudes,
        method=ReadoutScanMethod.SWEEP,
        submit_only=submit_only,
        live_html_output_dir=OUTPUT_ROOT,
        workflow_settings=workflow_settings,
    )
    return ReadoutAmplitudeSweepWorkflow(
        qubit_names=list(QUBIT_NAMES),
        profile=profile,
        task_manager=task_manager,
        settings=optimizer_settings,
    )


## Stage 1: Submit Kernel Experiments

Run this once to submit kernel-trace experiments for all configured amplitudes. Save the printed `run_dir` or `run_key` for Stage 2.

In [ ]:
optimizer = make_optimizer(
    amplitudes=AMPLITUDES,
    run_kernels=True,
    run_iq_blobs=False,
    submit_only=True,
)

optimizer.submit_kernel_stage()
RUN_KEY = optimizer.run_dir.name
RUN_DIR = optimizer.run_dir.resolve()

print("run_dir:", RUN_DIR)
print("run_key:", RUN_KEY)


[qratena] [2026.07.08 00:00:32] INFO     ProfileManager connected (db=QPUs, container=profiles)
[qratena] [2026.07.08 00:00:32] INFO     pull_profile: branch='main'
[qratena] [2026.07.08 00:00:32] INFO     pull_kernels: 20 qubit(s) → /Users/asafsolonnikov/Developer/Qarakal/workbench/data/qratena_kernel_traces
[qratena] [2026.07.08 00:00:36] INFO     ProfileManager disconnected
Readout optimization [------------------------------] 0/25 (  0.0%) current=0.005[readout workflow] kernels started
[qratena] [2026.07.08 00:00:36] INFO     using directly supplied profile
[qratena] [2026.07.08 00:00:36] INFO     Initialized ninja_chip_device QPU with 20 qubits and 32 couplers
[2026.07.08 00:00:36.173] INFO    Logging initialized from [Default inline config in laboneq.laboneq_logging] logdir is /Users/asafsolonnikov/Developer/Qarakal/workbench/laboneq_output/log
[2026.07.08 00:00:36.183] INFO    VERSION: laboneq 26.4.0
[2026.07.08 00:00:36.186] INFO    Connecting to data server at 127.0.0.1:8004


## Inspect Pending Folder

Use this to inspect the saved manifest and metadata for a pending or staged run.

In [34]:
run_dir = resolve_run_dir(RUN_KEY)
metadata = load_metadata(run_dir)
manifest = load_readout_task_manifest(run_dir)

print("run_dir:", run_dir)
print("run_key:", metadata.get("run_key", run_dir.name))
print("run_status:", manifest.get("run_status"))
print("qubits:", manifest.get("qubits"))
print("amplitudes:", manifest.get("amplitudes"))
print("task_count:", len(manifest.get("tasks", [])))
print("nodes:", sorted({task.get("node") for task in manifest.get("tasks", [])}))


run_dir: /Users/asafsolonnikov/Developer/Qarakal/workbench/data/readout_optimize/2026-07-07/23-56-33_sweep_q5
run_key: 23-56-33_sweep_q5
run_status: submitted_pending_results
qubits: ['q5']
amplitudes: [0.005, 0.041249999999999995, 0.0775, 0.11374999999999999, 0.15]
task_count: 5
nodes: ['kernels']


## Check Task Status Without Waiting

This updates `task_manifest.json` with task statuses and returns immediately.

In [35]:
run_dir = resolve_run_dir(RUN_KEY)
optimizer = make_optimizer(
    amplitudes=[],
    run_kernels=True,
    run_iq_blobs=False,
)

summary = optimizer.check_submitted_results(run_dir)

print(summary["message"])
print("counts:", summary["counts"])
print_task_keys("Pending task keys", summary["pending_task_keys"])
print_task_keys("Failed/cancelled task keys", summary["failed_task_keys"])


[qratena] [2026.07.07 23:57:31] INFO     ProfileManager connected (db=QPUs, container=profiles)
[qratena] [2026.07.07 23:57:31] INFO     pull_profile: branch='main'
[qratena] [2026.07.07 23:57:32] INFO     pull_kernels: 20 qubit(s) → /Users/asafsolonnikov/Developer/Qarakal/workbench/data/qratena_kernel_traces
[qratena] [2026.07.07 23:57:35] INFO     ProfileManager disconnected
All submitted readout tasks are complete; results are ready to collect.
All submitted readout tasks are complete; results are ready to collect.
counts: {'completed': 5, 'queued': 0, 'running': 0, 'failed': 0, 'cancelled': 0, 'unknown': 0}


## Stage 2: Collect Kernels And Submit IQ Blobs

Run this after the kernel tasks are finished. With `WAIT_FOR_RESULTS = False`, this submits IQ blobs and exits after updating the manifest. Run the status cell again later to monitor the IQ tasks.

In [36]:
run_dir = resolve_run_dir(RUN_KEY)
optimizer = make_optimizer(
    amplitudes=[],
    run_kernels=True,
    run_iq_blobs=False,
)

result = optimizer.collect_kernels_submit_iq_blobs(
    run_dir,
    wait_for_iq_results=WAIT_FOR_RESULTS,
    save_results=True,
)

if isinstance(result, dict) and "ready_to_collect" in result:
    print(result["message"])
    print("counts:", result["counts"])
else:
    print("Collected final optimizer results into:", run_dir)


[qratena] [2026.07.07 23:57:40] INFO     ProfileManager connected (db=QPUs, container=profiles)
[qratena] [2026.07.07 23:57:40] INFO     pull_profile: branch='main'
[qratena] [2026.07.07 23:57:40] INFO     pull_kernels: 20 qubit(s) → /Users/asafsolonnikov/Developer/Qarakal/workbench/data/qratena_kernel_traces
[qratena] [2026.07.07 23:57:43] INFO     ProfileManager disconnected
[qratena] [2026.07.07 23:57:43] INFO     using directly supplied profile
[qratena] [2026.07.07 23:57:43] INFO     Initialized ninja_chip_device QPU with 20 qubits and 32 couplers
[2026.07.07 23:57:43.923] INFO    Logging initialized from [Default inline config in laboneq.laboneq_logging] logdir is /Users/asafsolonnikov/Developer/Qarakal/workbench/laboneq_output/log
[2026.07.07 23:57:43.932] INFO    VERSION: laboneq 26.4.0
[2026.07.07 23:57:43.933] INFO    Connecting to data server at 127.0.0.1:8004
[2026.07.07 23:57:43.934] INFO    Connected to Zurich Instruments LabOne Data Server version 26.04.1.6 at 127.0.0.1:

## Collect IQ Results From An Already-Staged Run

Use this if Stage 2 already submitted IQ blob tasks in a previous notebook/script run and you only need to collect/analyze them now.

In [37]:
run_dir = resolve_run_dir(RUN_KEY)
optimizer = make_optimizer(
    amplitudes=[],
    run_kernels=False,
    run_iq_blobs=True,
)

result = optimizer.collect_staged_iq_results(
    run_dir,
    save_results=True,
    wait=WAIT_FOR_RESULTS,
)

if isinstance(result, dict) and "ready_to_collect" in result:
    print(result["message"])
    print("counts:", result["counts"])
else:
    print("Collected final optimizer results into:", run_dir)


[qratena] [2026.07.07 23:58:42] INFO     ProfileManager connected (db=QPUs, container=profiles)
[qratena] [2026.07.07 23:58:42] INFO     pull_profile: branch='main'
[qratena] [2026.07.07 23:58:43] INFO     pull_kernels: 20 qubit(s) → /Users/asafsolonnikov/Developer/Qarakal/workbench/data/qratena_kernel_traces
[qratena] [2026.07.07 23:58:46] INFO     ProfileManager disconnected
All submitted readout tasks are complete; results are ready to collect.
All submitted readout tasks are complete; results are ready to collect.
counts: {'completed': 10, 'queued': 0, 'running': 0, 'failed': 0, 'cancelled': 0, 'unknown': 0}


## Automatic Route For Existing Run Folder

This cell decides what to do based on `task_manifest.json`:

- kernel-only manifest: collect kernels and submit IQ blobs;
- staged IQ manifest: collect IQ results;
- old non-staged manifest: collect submitted results using the legacy path.

In [ ]:
run_dir = resolve_run_dir(RUN_KEY)
manifest = load_readout_task_manifest(run_dir)
optimizer = make_optimizer(
    amplitudes=[],
    run_kernels=True,
    run_iq_blobs=False,
)

if is_staged_kernel_run(manifest):
    result = optimizer.collect_kernels_submit_iq_blobs(
        run_dir,
        wait_for_iq_results=WAIT_FOR_RESULTS,
        save_results=True,
    )
elif is_staged_iq_run(manifest):
    status = None if WAIT_FOR_RESULTS else optimizer.check_submitted_results(run_dir)
    result = optimizer.collect_staged_iq_results(
        run_dir,
        save_results=True,
        wait=WAIT_FOR_RESULTS or bool(status and status["ready_to_collect"]),
    )
    if status and not status["ready_to_collect"]:
        result = status
else:
    status = None if WAIT_FOR_RESULTS else optimizer.check_submitted_results(run_dir)
    result = optimizer.collect_submitted_results(
        run_dir,
        save_results=True,
        wait=WAIT_FOR_RESULTS or bool(status and status["ready_to_collect"]),
    )
    if status and not status["ready_to_collect"]:
        result = status

if isinstance(result, dict) and "ready_to_collect" in result:
    print(result["message"])
    print("counts:", result["counts"])
else:
    print("Done:", run_dir)


[qratena] [2026.07.07 23:50:59] INFO     ProfileManager connected (db=QPUs, container=profiles)
[qratena] [2026.07.07 23:50:59] INFO     pull_profile: branch='main'
[qratena] [2026.07.07 23:51:00] INFO     pull_kernels: 20 qubit(s) → /Users/asafsolonnikov/Developer/Qarakal/workbench/data/qratena_kernel_traces
[qratena] [2026.07.07 23:51:03] INFO     ProfileManager disconnected
All submitted readout tasks are complete; results are ready to collect.
All submitted readout tasks are complete; results are ready to collect.
counts: {'completed': 10, 'queued': 0, 'running': 0, 'failed': 0, 'cancelled': 0, 'unknown': 0}


## Validation: Current IQ Fidelity vs Optimized IQ Fidelity

The validator is also staged.

Stage 1 submits two kinds of tasks:

- current-profile IQ blobs only, without rerunning the full readout workflow;
- optimized-profile kernel tasks, because optimized IQ blobs must be compiled from the newly generated kernels.

Stage 2 collects/analyzes the optimized kernels, saves the kernel files, submits optimized IQ blobs for both reset conditions, and later collects the optimized IQ results.

The final validation folder contains two plots: one for current-profile fidelities and one for optimized-profile fidelities. Each plot shows fidelity by qubit with and without active reset.

In [ ]:
VALIDATION_RUN_KEY = "23-45-29_sweep_q5"  # folder name or full path from validation submit
VALIDATION_OUTPUT_ROOT = Path("data") / "readout_optimizer_validation_iq_blobs"
USE_PER_QUBIT_BEST_AMPLITUDE = True


In [ ]:
from optimize.readout.scripts import validate_optimizer_iq_blobs as validation
ACTIVE_RESET_NUM = 5

def configure_validation_module() -> None:
    validation.PROFILE_NAME = PROFILE_NAME
    validation.OUTPUT_ROOT = VALIDATION_OUTPUT_ROOT
    validation.ACTIVE_RESET_NUM = ACTIVE_RESET_NUM
    validation.LOW_PRIORITY_TASKS = LOW_PRIORITY_TASKS
    validation.WAIT_FOR_RESULTS = WAIT_FOR_RESULTS
    validation.USE_PER_QUBIT_BEST_AMPLITUDE = USE_PER_QUBIT_BEST_AMPLITUDE


configure_validation_module()


### Submit Validation Stage 1

Run this after the optimizer has final results in `RUN_KEY`. It submits current-profile IQ blobs and optimized-profile kernels.

In [ ]:
configure_validation_module()
validation_run_dir = validation.submit_validation(RUN_KEY)
print("validation_run_dir:", validation_run_dir)
print("validation_run_key:", validation_run_dir.name)


FileNotFoundError: Missing required artifact: /Users/asafsolonnikov/Developer/Qarakal/workbench/data/readout_optimize/2026-07-07/23-45-29_sweep_q5/summary.json

### Check Validation Status

Use this for either validation stage. It updates the validation `task_manifest.json` and exits without waiting.

In [ ]:
configure_validation_module()
validation_summary = validation.check_validation_results(VALIDATION_RUN_KEY)
validation_summary


### Continue Or Collect Validation

Run this after validation Stage 1 tasks finish. With `WAIT_FOR_RESULTS = False`, the first successful run collects current IQ results, collects optimized kernels, submits optimized IQ blobs, and exits. Run it again later to collect optimized IQ results and write the final reports/plots.

In [ ]:
configure_validation_module()
validation_result = validation.collect_validation_results(VALIDATION_RUN_KEY)
validation_result


### Inspect Validation Outputs

Final validation writes two active-reset comparison plots:

- `current_readout_fidelities_active_reset_comparison.png`
- `optimized_readout_fidelities_active_reset_comparison.png`

In [ ]:
from IPython.display import display, Image

configure_validation_module()
validation_dir = validation.resolve_validation_run_dir(VALIDATION_RUN_KEY)
for path in sorted(validation_dir.iterdir()):
    print(path.name + ("/" if path.is_dir() else ""))

current_plot = validation_dir / "current_readout_fidelities_active_reset_comparison.png"
optimized_plot = validation_dir / "optimized_readout_fidelities_active_reset_comparison.png"
if current_plot.exists():
    display(Image(filename=str(current_plot)))
if optimized_plot.exists():
    display(Image(filename=str(optimized_plot)))


## Inspect Final Results

After final collection, the run folder should contain the normal optimizer artifacts plus `kernel_files/`.

In [ ]:
run_dir = resolve_run_dir(RUN_KEY)

for path in sorted(run_dir.iterdir()):
    print(path.name + ("/" if path.is_dir() else ""))

summary_path = run_dir / "summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print("\nBest amplitudes:")
    print(json.dumps(summary.get("best_amplitudes", summary), indent=2, default=str))
